# Z3-19 — Optimisation multi-objectif avec Z3Trois modes : **lexicographique** (priorité stricte), **Pareto** (front de solutions non-dominées), **somme pondérée** (compromis scalaire). On montre les trois sur un même problème jouet (allocation de budget à 3 projets A/B/C), puis sur un front de Pareto explicite, et enfin sur une combinaison linéaire.**Prérequis** : Z3 ≥ 4.10 (présente localement : 4.16.0), `z3-solver` PyPI.**Public** : cours SMT, post-`Z3-18-Sudoku-Modes-Python`.**Objectifs** : savoir choisir entre lex / Pareto / weighted en fonction du cahier des charges.

## 1. Trois modes d'optimisation multi-objectif

Z3 propose `Optimize()` pour les problèmes satisfiables avec critères. Quand plusieurs objectifs s'affrontent, on a trois familles de réponses :

| Mode | Quand l'utiliser | Sémantique | Réponse Z3 |
|-------|------------------|-----------|------------|
| **Lexicographique** | priorité stricte entre objectifs (latence d'abord, coût ensuite) | optimise obj1 ; à obj1 optimal, optimise obj2 | `opt.minimize(o1); opt.maximize(o2)` |
| **Pareto** | aucun compromis acceptable a priori, on veut tous les équilibres | maximise le nombre d'objectifs non-dominés | énumération par `domine()` (cf cellule 2) |
| **Somme pondérée** | compromis scalaire connu, agrégation linéaire | minimise Σ wᵢ · objᵢ | `opt.minimize(w1*o1 + w2*o2)` |

On explore les trois avec un même problème jouet (allocation de budget à 3 projets), puis on vérifie la **non-dominance** sur le front de Pareto.

In [1]:
from z3 import *

# Trois projets A, B, C avec coût (k€) et impact (utilisateurs servis, en milliers).
# Budget total : 12 k€. Contrainte additionnelle : au moins 1 projet retenu
# (sans elle, la solution lex retournerait trivialement tous False pour coût = 0).
cout_A, cout_B, cout_C = 5, 4, 7
impact_A, impact_B, impact_C = 8, 6, 12

x, y, z = Bools('x y z')
cout_total = If(x, cout_A, 0) + If(y, cout_B, 0) + If(z, cout_C, 0)
impact_total = If(x, impact_A, 0) + If(y, impact_B, 0) + If(z, impact_C, 0)
selection_count = Sum([If(x, 1, 0), If(y, 1, 0), If(z, 1, 0)])

opt = Optimize()
opt.add(cout_total <= 12)
opt.add(selection_count >= 1)            # au moins 1 projet retenu
opt.minimize(cout_total)                 # priorité 1 : on minimise le coût
opt.maximize(impact_total)               # priorité 2 : à coût minimal, on maximise l'impact

assert opt.check() == sat
m = opt.model()
selection = [str(v) for v in (x, y, z) if is_true(m.eval(v, model_completion=True))]
cout_val = m.eval(cout_total, model_completion=True).as_long()
impact_val = m.eval(impact_total, model_completion=True).as_long()
print(f"Lex min : sélection = {selection}")
print(f"  coût total   = {cout_val} k€")
print(f"  impact total = {impact_val} k utilisateurs")

Lex min : sélection = ['y']
  coût total   = 4 k€
  impact total = 6 k utilisateurs


### 1.1 Interprétation — Lexicographique

Avec la contrainte « au moins 1 projet retenu » (sans elle Z3 retournerait `[]`, coût 0, optimum trivial), le solveur trouve le **plus petit coût non nul** : `B seul` coûte 4 k€. À coût égal, le solveur maximise l'impact. Comme aucun autre sous-ensemble ne coûte 4 k€ avec un meilleur impact, la sélection est `B seul`.

**Point pédagogique** : la lex n'est **pas** un compromis — Z3 choisit 4 k€ même si `B + C` à 11 k€ apporte 12k utilisateurs de plus. Si l'utilisateur veut équilibrer, il faut passer en **somme pondérée** avec un coefficient sur l'impact (cf cellule 3).

In [2]:
# Pareto : on cherche toutes les solutions non-dominées sur (coût, impact).
# Une solution A domine B si coût(A) <= coût(B) et impact(A) >= impact(B) avec au moins une stricte.
from itertools import product

solutions = []
for sx, sy, sz in product([False, True], repeat=3):
    c = (sx * cout_A + sy * cout_B + sz * cout_C)
    i = (sx * impact_A + sy * impact_B + sz * impact_C)
    if 0 < c <= 12:  # budget respecté, au moins un projet
        solutions.append((c, i, (sx, sy, sz)))

def domine(a, b):
    """A domine B si A est <= sur coût et >= sur impact, avec au moins une stricte."""
    return (a[0] <= b[0] and a[1] >= b[1]) and (a[0] < b[0] or a[1] > b[1])

non_dominees = []
for s in solutions:
    if not any(domine(t, s) for t in solutions if t is not s):
        non_dominees.append(s)
non_dominees.sort()

print("Front de Pareto (coût k€, impact k utilisateurs, sélection) :")
for c, i, sel in non_dominees:
    noms = [n for n, v in zip("ABC", sel) if v]
    print(f"  coût={c:2d} k€, impact={i:2d} k  ->  {','.join(noms) if noms else '∅'}")
print(f"\nNombre de solutions Pareto-optimales : {len(non_dominees)}")

Front de Pareto (coût k€, impact k utilisateurs, sélection) :
  coût= 4 k€, impact= 6 k  ->  B
  coût= 5 k€, impact= 8 k  ->  A
  coût= 7 k€, impact=12 k  ->  C
  coût= 9 k€, impact=14 k  ->  A,B
  coût=11 k€, impact=18 k  ->  B,C
  coût=12 k€, impact=20 k  ->  A,C

Nombre de solutions Pareto-optimales : 6


### 1.2 Interprétation — Front de Pareto

Le front contient **6 solutions** (sur 7 sous-ensembles non-vides ; `A + B + C` à 16 k€ est hors budget). Aucune n'est *objectivement* meilleure :

- `B seul` (4/6) : coût minimal, impact minimal
- `A seul` (5/8) : pas cher, faible impact
- `C seul` (7/12) : bon impact, coût moyen
- `A + B` (9/14) : compromis intermédiaire
- `B + C` (11/18) : très bon impact
- `A + C` (12/20) : impact maximal (coût maximal aussi)

**Point pédagogique** : aucun sous-ensemble n'est dominé sur ce problème — chaque projet apporte un couple (coût, impact) **non dominé** (le rapport coût/impact de `B` à 0.67 k€/k util. est le meilleur, mais aucun autre sous-ensemble n'a simultanément coût inférieur ET impact supérieur). Pour un décideur, c'est ici que le passage au **somme pondérée** (avec un `w_impact` qui traduit une valeur monétaire de l'utilisateur) fait sens.

In [3]:
# Compromis scalaire : on attribue une valeur monétaire à l'impact (1k utilisateurs = 0.5 k€).
# Score = coût - 0.5 * impact. On minimise le score.
opt2 = Optimize()
opt2.add(cout_total <= 12)
opt2.add(selection_count >= 1)
score = cout_total - 0.5 * impact_total
opt2.minimize(score)

assert opt2.check() == sat
m2 = opt2.model()
sel2 = [str(v) for v in (x, y, z) if is_true(m2.eval(v, model_completion=True))]
cout_v2 = m2.eval(cout_total, model_completion=True).as_long()
imp_v2 = m2.eval(impact_total, model_completion=True).as_long()
score_v2 = m2.eval(score, model_completion=True).as_long()
print(f"Somme pondérée (w_impact=0.5 k€/k util.) : sélection = {sel2}")
print(f"  coût total   = {cout_v2} k€")
print(f"  impact total = {imp_v2} k utilisateurs")
print(f"  score        = {score_v2}")

# Variation : w_impact = 1.0 (impact vaut son pesant en k€)
opt3 = Optimize()
opt3.add(cout_total <= 12)
opt3.add(selection_count >= 1)
score3 = cout_total - 1.0 * impact_total
opt3.minimize(score3)
assert opt3.check() == sat
m3 = opt3.model()
sel3 = [str(v) for v in (x, y, z) if is_true(m3.eval(v, model_completion=True))]
score_v3 = m3.eval(score3, model_completion=True).as_long()
print(f"\nSomme pondérée (w_impact=1.0) : sélection = {sel3}")
print(f"  score = {score_v3}")

Somme pondérée (w_impact=0.5 k€/k util.) : sélection = ['z']
  coût total   = 7 k€
  impact total = 12 k utilisateurs
  score        = 1

Somme pondérée (w_impact=1.0) : sélection = ['x', 'z']
  score = -8


### 1.3 Interprétation — Sensibilité aux poids

Avec `w_impact = 0.5` (1k utilisateurs vaut 500 €), le solveur retient `C seul` (coût 7, impact 12, score = 7 - 6 = 1). Avec `w_impact = 1.0` (1k utilisateurs vaut 1 k€), il retient `A + C` (coût 12, impact 20, score = 12 - 20 = -8). La sélection **change** entre les deux poids : on n'est PAS dans une zone de stabilité.

**Point pédagogique** : la **non-stabilité** du choix entre `w = 0.5` et `w = 1.0` est un signal que le décideur n'a pas de taux d'échange bien calibré. Pour stabiliser, il faut soit descendre à `w_impact < 0.5` (où `C seul` reste optimal), soit monter à `w_impact > 1.0` (où `A + C` reste optimal), soit **interpoler** explicitement entre les deux. La méthode propre : tracer le **score** des 6 solutions Pareto en fonction de `w_impact` et trouver les points de croisement — c'est exactement le **diagramme de tradeoff**.

In [4]:
# E1 — Lexicographique sur 4 variables
#
# Reprendre le problème ci-dessus avec 4 projets A, B, C, D de coûts (3, 5, 6, 8) k€ et
# d'impacts (4, 7, 10, 13) k utilisateurs, budget 15 k€.
#
# Trouver la sélection lexicographique :
#   priorité 1 = minimiser le coût total
#   priorité 2 = maximiser l'impact total
#
# Afficher : la sélection, le coût total, l'impact total.

def lex_min_4_vars():
    # TODO etudiant : utiliser Optimize() + minimize() + maximize() en cascade
    # Penser à utiliser m.eval() pour lire les valeurs (pas m[expr])
    # et Sum([If(b, 1, 0), ...]) pour la contrainte de cardinalité
    pass

lex_min_4_vars()

In [5]:
# E2 — Front de Pareto sur 4 variables
#
# À partir des 4 projets ci-dessus, calculer le **front de Pareto** sur (coût, impact)
# dans la limite du budget 15 k€.
#
# Afficher : la liste des solutions non-dominées, triées par coût croissant, et le
# nombre total de solutions Pareto-optimales.

def pareto_front_4_vars():
    # TODO etudiant : enumerer les 2^4 = 16 sous-ensembles, filtrer (0 < cout <= 15),
    # appliquer la definition de domination (cf cellule 2 ci-dessus)
    pass

pareto_front_4_vars()

In [6]:
# E3 — Somme pondérée avec 4 poids
#
# Tester 4 jeux de poids (w_impact ∈ {0.2, 0.5, 1.0, 2.0}) sur le problème à 4 projets.
#
# Pour chaque poids, afficher la sélection optimale et son score.
# Observer la **stabilité** : à partir de quel poids la sélection change-t-elle ?

def weighted_sum_4_weights():
    # TODO etudiant : boucle sur 4 poids, pour chacun construire un Optimize() et
    # collecter (selection, score, w_impact) dans une liste
    pass

weighted_sum_4_weights()

## 2. Quand utiliser chaque mode- **Lexicographique** : priorité **non-négociable** entre objectifs (sécurité > coût,   latence > énergie). Le décideur sait ordonner les critères, pas les pondérer.- **Pareto** : aucun compromis acceptable, on veut **tous** les équilibres pour arbitrage   humain (conception de systèmes, négociation). Coûteux en énumération si le front est grand.- **Somme pondérée** : compromis scalaire **connu et stable** (coût financier direct).   Simple, traçable, mais perd la structure du front (un seul point).**Recommandation pratique** :1. Si l'utilisateur sait **ordonner** les critères → lex.2. Si l'utilisateur ne sait pas pondérer mais veut **tous** les équilibres → Pareto.3. Si l'utilisateur a un **taux d'échange** clair → somme pondérée.4. En cas de doute : explorer le **front de Pareto** puis vérifier la **stabilité** du    choix sur une plage de poids (cf E3).**Vers la suite** : `Z3-06-Advanced-Optimization-Python.ipynb` (introduction à `Optimize()`, MaxSAT souple, Pareto/box) pour le tour d'horizon, ou `Z3-16e-Meal-Planner-Optimize-Python.ipynb` pour un cas d'usage complet (menu multi-objectif) qui réutilise les trois modes.